# Mouvement Brownien Géométrique (GBM)

Ce notebook implémente :
- Le mouvement brownien standard
- Le GBM via la méthode d'Euler-Maruyama
- Le GBM via le lemme d'Itô (solution exacte)
- Le pricing Monte-Carlo d'une option Call européenne avec convergence en O(1/√n)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Mouvement Brownien Standard

In [ ]:
# Paramètres
T = 1.0       # horizon de temps
N = 252       # nombre de pas 
n_paths = 10  # nombre de trajectoires

dt = T / N
t = np.linspace(0, T, N+1)

# Simulation du mouvement brownien
np.random.seed(42)
dW = np.random.normal(0, np.sqrt(dt), size=(n_paths, N))
W = np.hstack([np.zeros((n_paths, 1)), np.cumsum(dW, axis=1)])

plt.figure(figsize=(10, 5))
for i in range(n_paths):
    plt.plot(t, W[i], lw=1)
plt.title('Mouvement Brownien Standard')
plt.xlabel('Temps')
plt.ylabel('W(t)')
plt.grid(True)
plt.tight_layout()
plt.show()

## 2. GBM — Méthode d'Euler-Maruyama

L'EDS du GBM est :

$$dS_t = \mu S_t \, dt + \sigma S_t \, dW_t$$

Discrétisation d'Euler-Maruyama :

$$S_{t+dt} = S_t + \mu S_t \, dt + \sigma S_t \, dW_t$$

In [ ]:
# Paramètres du GBM
S0    = 100.0   # prix initial
mu    = 0.05    # drift (5%)
sigma = 0.20    # volatilité (20%)

# Euler-Maruyama
S_euler = np.zeros((n_paths, N+1))
S_euler[:, 0] = S0

np.random.seed(42)
for i in range(N):
    dW_i = np.random.normal(0, np.sqrt(dt), size=n_paths)
    S_euler[:, i+1] = S_euler[:, i] + mu * S_euler[:, i] * dt + sigma * S_euler[:, i] * dW_i

plt.figure(figsize=(10, 5))
for i in range(n_paths):
    plt.plot(t, S_euler[i], lw=1)
plt.title('GBM — Euler-Maruyama')
plt.xlabel('Temps')
plt.ylabel('S(t)')
plt.grid(True)
plt.tight_layout()
plt.show()

## 3. GBM — Solution Exacte via le Lemme d'Itô

Par le lemme d'Itô appliqué à $\ln(S_t)$ :

$$S_t = S_0 \exp\left(\left(\mu - \frac{\sigma^2}{2}\right)t + \sigma W_t\right)$$

In [ ]:
# Solution exacte (lemme d'Itô)
np.random.seed(42)
dW_exact = np.random.normal(0, np.sqrt(dt), size=(n_paths, N))
W_exact = np.hstack([np.zeros((n_paths, 1)), np.cumsum(dW_exact, axis=1)])

S_ito = S0 * np.exp((mu - 0.5 * sigma**2) * t + sigma * W_exact)

plt.figure(figsize=(10, 5))
for i in range(n_paths):
    plt.plot(t, S_ito[i], lw=1)
plt.title('GBM — Solution Exacte (Lemme d\'Itô)')
plt.xlabel('Temps')
plt.ylabel('S(t)')
plt.grid(True)
plt.tight_layout()
plt.show()

## 4. Comparaison Euler vs Itô

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(t, S_euler[0], label='Euler-Maruyama', lw=1.5)
plt.plot(t, S_ito[0],   label='Solution exacte (Itô)', lw=1.5, linestyle='--')
plt.title('Comparaison : Euler-Maruyama vs Solution Exacte (1 trajectoire)')
plt.xlabel('Temps')
plt.ylabel('S(t)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 5. Pricing Monte-Carlo — Option Call Européenne

Pour une option Call européenne de strike $K$ et maturité $T$, le prix est :

$$C = e^{-rT} \mathbb{E}[\max(S_T - K, 0)]$$

On estime cette espérance par la moyenne empirique sur $n$ simulations.  
L'erreur d'estimation suit une loi en $O(1/\sqrt{n})$ par le TCL.

In [ ]:
# Paramètres
K = 105.0   # strike
r = 0.03    # taux sans risque
T = 1.0

np.random.seed(0)
n_simulations = 100000

# Simulation des prix finaux S_T
Z = np.random.normal(0, 1, n_simulations)
S_T = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)

# Payoff et prix actualisé
payoffs = np.maximum(S_T - K, 0)
prix_call = np.exp(-r * T) * np.mean(payoffs)
print(f"Prix Call Monte-Carlo  : {prix_call:.4f}")

# Intervalle de confiance à 95%
std_payoffs = np.std(payoffs)
IC_95 = 1.96 * std_payoffs / np.sqrt(n_simulations)
print(f"Intervalle de confiance 95% : [{prix_call - IC_95:.4f}, {prix_call + IC_95:.4f}]")

## 6. Convergence en O(1/√n)

In [ ]:
# Convergence du prix Monte-Carlo en fonction de n
np.random.seed(0)
n_values = np.logspace(1, 5, 50).astype(int)
prix_par_n = []

Z_all = np.random.normal(0, 1, max(n_values))
S_T_all = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z_all)
payoffs_all = np.maximum(S_T_all - K, 0)

for n in n_values:
    prix_n = np.exp(-r * T) * np.mean(payoffs_all[:n])
    prix_par_n.append(prix_n)

plt.figure(figsize=(10, 5))
plt.semilogx(n_values, prix_par_n, lw=1.5, label='Prix estimé')
plt.axhline(y=prix_call, color='red', linestyle='--', label=f'Prix convergé : {prix_call:.4f}')
plt.title('Convergence du prix Monte-Carlo en O(1/√n)')
plt.xlabel('Nombre de simulations (n)')
plt.ylabel('Prix Call estimé')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 7. Erreur d'estimation vs 1/√n

In [ ]:
erreurs = np.abs(np.array(prix_par_n) - prix_call)
courbe_theorique = erreurs[0] * np.sqrt(n_values[0]) / np.sqrt(n_values)

plt.figure(figsize=(10, 5))
plt.loglog(n_values, erreurs, lw=1.5, label='Erreur empirique')
plt.loglog(n_values, courbe_theorique, lw=1.5, linestyle='--', label='Courbe théorique O(1/√n)')
plt.title('Erreur Monte-Carlo vs O(1/√n)')
plt.xlabel('Nombre de simulations (n)')
plt.ylabel('Erreur absolue')
plt.legend()
plt.grid(True, which='both')
plt.tight_layout()
plt.show()